In [10]:
import os
import re
import json
import shutil
import subprocess
from pathlib import Path
from ipywidgets import FileUpload
from IPython.display import Video, display

In [21]:
uploader = FileUpload(accept='.wav,.flac,.mp3,.mp4', multiple=True)

display(uploader)

FileUpload(value=(), accept='.wav,.flac,.mp3,.mp4', description='Upload', multiple=True)

In [4]:
def check_ffmpeg():
    result = subprocess.run(
        ["ffmpeg", "-version"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError("FFmpeg is not available.")

    print(result.stdout.splitlines()[0])


check_ffmpeg()

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


In [5]:
def parse_model_script(raw_json):
    match = re.search(
        r'"full_script"\s*:\s*"(.*)"',
        raw_json,
        re.DOTALL
    )

    if not match:
        raise ValueError("Cannot find 'full_script'")

    full_script = match.group(1)

    full_script = full_script.replace("\\n", "\n")

    pattern = re.compile(
        r"\[\s*([\d.]+)s\s*-\s*([\d.]+)s\s*\]:\s*(.*)"
    )

    segments = []

    for line in full_script.splitlines():

        line = line.strip()

        if not line:
            continue

        match = pattern.match(line)

        if not match:
            print(f"Skipping invalid line: {line}")
            continue

        start = float(match.group(1))
        end = float(match.group(2))
        text = match.group(3).strip()

        if end <= start:
            print(
                f"Skipping invalid timestamp: "
                f"{start} -> {end}"
            )
            continue

        segments.append({
            "start": start,
            "end": end,
            "duration": round(end - start, 3),
            "text": text
        })

    return segments

def cut_video_segments(
    video_path,
    segments,
    output_dir,
    video_codec="libx264",
    audio=False
):
    video_path = Path(video_path)
    output_dir = Path(output_dir)

    if not video_path.exists():
        raise FileNotFoundError(
            f"Video not found: {video_path}"
        )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    results = []

    for index, segment in enumerate(segments, start=1):

        start = segment["start"]
        end = segment["end"]
        duration = end - start

        output_path = (
            output_dir /
            f"segment_{index:03d}.mp4"
        )

        command = [
            "ffmpeg",
            "-y",

            # Accurate seeking
            "-ss", str(start),
            "-i", str(video_path),

            "-t", str(duration),

            # Video only
            "-an",

            # Re-encode for accurate boundaries
            "-c:v", video_codec,

            # Quality
            "-preset", "fast",
            "-crf", "18",

            # MP4 compatibility
            "-movflags", "+faststart",

            str(output_path)
        ]

        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode != 0:
            print(f"FAILED: segment {index}")
            print(result.stderr)
            continue

        result_data = {
            "index": index,
            "start": start,
            "end": end,
            "duration": duration,
            "text": segment["text"],
            "path": str(output_path)
        }

        results.append(result_data)

        print(
            f"[{index:03d}] "
            f"{start:.2f}s → {end:.2f}s "
            f"| {output_path.name}"
        )

    return results

def save_manifest(results, output_dir):
    manifest_path = Path(output_dir) / "manifest.json"

    manifest = {
        "source_video": str(VIDEO_PATH),
        "audio": False,
        "segment_count": len(results),
        "segments": results
    }

    with open(
        manifest_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            manifest,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Manifest saved: {manifest_path}")

    return manifest_path

def create_zip(output_dir):
    output_dir = Path(output_dir)

    zip_base = output_dir.parent / output_dir.name

    zip_path = shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=output_dir
    )

    print(f"ZIP created: {zip_path}")

    return zip_path

print("done")

In [7]:
# =========================
# CONFIGURATION
# =========================

VIDEO_PATH = "/kaggle/working/Download.mp4"

OUTPUT_DIR = "/kaggle/working/video_segments"

CREATE_ZIP = False

raw_model_output = r'''
"full_script": "[0.00s - 0.84s]: Anh muốn nghỉ việc?\n[0.84s - 1.40s]: Không thì đâu anh?\n[1.40s - 2.12s]: Tại sao không?\n[2.12s - 5.68s]: Trong hợp đồng ghi rõ là nếu mà công việc không giống với việc mô tả thì được đơn phương nghỉ việc\n[5.68s - 6.84s]: Đúng, nhưng mà riêng anh thì không\n[6.84s - 7.44s]: Tại sao?\n[7.44s - 8.92s]: Tại vì anh là Founder\n[8.92s - 11.40s]: Anh ơi anh ráng đây, đây là lần thứ 3 suy nghĩ trong ngàn ngày đó\n[11.40s - 13.00s]: Anh mà nghĩ ra anh trả lương cho tụi em\n[13.00s - 14.12s]: Mà anh mệt\n[14.12s - 17.12s]: Ok, hôm nay chúng ta sẽ họp để tìm ra phương án mới dành cho doanh nghiệp\n[17.12s - 17.92s]: Các anh chị\n[19.40s - 20.72s]: Ai rốt nước vậy ngưng coi\n[23.36s - 24.60s]: Ai rốt nước vậy ngưng coi\n[26.20s - 27.60s]: Tôi nói tắt nước\n[27.60s - 30.00s]: Tắt nước đây\n[30.00s - 32.26s]: Trời ơi nước ngờ hơn tắt đi mẹ\n[32.26s - 35.98s]: Rồi đó là cái hình của\n[35.98s - 37.18s]: Anh là ai vậy\n[37.18s - 38.04s]: Hôm nay còn sao vậy\n[38.04s - 39.00s]: Chuyện này là sao\n[39.00s - 39.98s]: Bình tĩnh đi Dung\n[39.98s - 41.10s]: Còn đang làm mọi người sợ đó\n[41.10s - 41.46s]: Dung\n[41.46s - 42.42s]: Dung nào\n[42.42s - 48.62s]: Cậu ổn không vậy\n[48.62s - 52.02s]: Dạ chị ổn không chị\n[52.02s - 52.96s]: Dạ chị ổn mà\n[52.96s - 54.06s]: Bộ sáng chị không ăn sáng hả\n[54.06s - 55.32s]: Đâu chị có đánh mì theo\n[55.32s - 57.40s]: Hợp đồng đối tác nước ngoài quan trọng\n[57.40s - 58.40s]: Dạ ok sếp\n[62.40s - 64.40s]: Aaaaaa không đừng xui xẻo\n[64.40s - 67.40s]: Xui xẻo không phải là cách, xui viện mới là cách\n[67.40s - 69.40s]: Viên sủi Bocales vì tăng lực sảng khoái\n[69.40s - 72.40s]: Chỉ cần 1 viên mỗi buổi sáng là chị sẽ có đủ Vitamin C và Vitamin nhóm B\n[72.40s - 75.40s]: Giúp chị duy trì sự tỉnh táo bền bỉ suốt cả ngày\n[75.40s - 78.40s]: Hương vị nước tăng lực đặc trưng nhưng mà lạnh mạnh hơn nhờ sử dụng đường ăn chiên nha\n[78.40s - 81.40s]: An tâm sức khỏe, tiện lợi, giá càng thêm hay, chị cầm lấy đi\n[81.40s - 84.40s]: Thì ra hôm nay chị mệt mỏi mà tưởng tụ tụm lum hết trơn á\n[84.40s - 88.12s]: Đúng rồi, với Pocales Multi Tropical chị sẽ luôn là chính mình mỗi ngày mà\n[88.12s - 89.04s]: Ủa sao em xin nghỉ vậy?\n[89.20s - 91.00s]: Thì em làm quảng cáo, sếp dặn em ghi là\n[91.00s - 93.10s]: Thật phẩm giúp bạn mỗi ngày đều sẵn khoái\n[93.10s - 93.88s]: Rồi em ghi sao?\n[94.04s - 94.80s]: Em ghi thiếu chữ khoái\n"
'''
print("done")

In [9]:
segments = parse_model_script(raw_model_output)

print(f"Parsed: {len(segments)} segments")

print(segments[0])

results = cut_video_segments(
    video_path=VIDEO_PATH,
    segments=segments,
    output_dir=OUTPUT_DIR
)

manifest_path = save_manifest(
    results,
    OUTPUT_DIR
)

if CREATE_ZIP:
    zip_path = create_zip(OUTPUT_DIR)

Parsed: 46 segments
{'start': 0.0, 'end': 0.84, 'duration': 0.84, 'text': 'Anh muốn nghỉ việc?'}
[001] 0.00s → 0.84s | segment_001.mp4
[002] 0.84s → 1.40s | segment_002.mp4
[003] 1.40s → 2.12s | segment_003.mp4
[004] 2.12s → 5.68s | segment_004.mp4
[005] 5.68s → 6.84s | segment_005.mp4
[006] 6.84s → 7.44s | segment_006.mp4
[007] 7.44s → 8.92s | segment_007.mp4
[008] 8.92s → 11.40s | segment_008.mp4
[009] 11.40s → 13.00s | segment_009.mp4
[010] 13.00s → 14.12s | segment_010.mp4
[011] 14.12s → 17.12s | segment_011.mp4
[012] 17.12s → 17.92s | segment_012.mp4
[013] 19.40s → 20.72s | segment_013.mp4
[014] 23.36s → 24.60s | segment_014.mp4
[015] 26.20s → 27.60s | segment_015.mp4
[016] 27.60s → 30.00s | segment_016.mp4
[017] 30.00s → 32.26s | segment_017.mp4
[018] 32.26s → 35.98s | segment_018.mp4
[019] 35.98s → 37.18s | segment_019.mp4
[020] 37.18s → 38.04s | segment_020.mp4
[021] 38.04s → 39.00s | segment_021.mp4
[022] 39.00s → 39.98s | segment_022.mp4
[023] 39.98s → 41.10s | segment_023.mp

In [20]:
with open(
    "/kaggle/working/video_segments/manifest.json",
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

segment = manifest["segments"][35]

print(f'File: {segment["path"]}')
print(f'Time: {segment["start"]:.2f}s → {segment["end"]:.2f}s')
print(f'Text: {segment["text"]}')

display(
    Video(
        segment["path"],
        embed=True
    )
)

File: /kaggle/working/video_segments/segment_036.mp4
Time: 69.40s → 72.40s
Text: Chỉ cần 1 viên mỗi buổi sáng là chị sẽ có đủ Vitamin C và Vitamin nhóm B
